In [142]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [143]:
words = open('names.txt','r').read().splitlines()

In [144]:
chars = sorted(list(set("".join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [274]:
# block_size=3
# X,y=[],[]
# for w in words:
#     print(w)
#     context = [0]*3
#     for ch in w+'.':
#         ix = stoi[ch]
#         X.append(context)
#         y.append(ix)
#         # print(context, ch)
#         print(''.join(itos[i] for i in context), '--->>', ch)
#         context = context[1:]+[ix]

# X=torch.tensor(X)
# y = torch.tensor(y)

import random
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

def dataset(words):
    block_size=3
    X,y = [],[]
    for w in words:
        context = [0]*block_size
        for ch in w+'.':
            ix = stoi[ch]
            X.append(context)
            y.append(ix)
            context = context[1:]+[ix]
    
    X = torch.tensor(X)
    y = torch.tensor(y)
    print(X.shape,y.shape)
    return X,y

Xtr,ytr = dataset(words[:n1])
Xdev,ydev = dataset(words[n1:n2])
Xte,yte = dataset(words[n2:])


torch.Size([182563, 3]) torch.Size([182563])
torch.Size([22823, 3]) torch.Size([22823])
torch.Size([22760, 3]) torch.Size([22760])


In [193]:
X.shape, y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [342]:
g = torch.Generator().manual_seed(2)
C = torch.randn((27,100), generator=g)
W1 = torch.randn((300,200), generator=g)
b1 = torch.randn(200,generator=g)
W2 = torch.randn((200,27), generator=g)
b2 = torch.randn(27,generator=g)
parameters = [C,W1,b1,W2,b2]

In [343]:
for p in parameters:
    p.requires_grad=True

In [ ]:
ls,lri=[],[]
for i in range(200):
    ix = torch.randint(0,Xtr.shape[0], (32,))
    # forward pass
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1,300)@W1+b1)
    # print(loss.item())
    logits = h@W2+b2
    loss=F.cross_entropy(logits,ytr[ix])
    for p in parameters:
        p.grad=None
    loss.backward()
    # lr= lrs[i]
    lr = 0.1 if i<100000 else 0.01
    for p in parameters:
        p.data+=-0.001*p.grad
    # ls.append(loss.item())
    # lri.append(lr)
print(loss.item())

2.688620090484619


In [348]:
emb = C[Xdev]
h = torch.tanh(emb.view(-1,300)@W1+b1)
logits = h@W2+b2
loss=F.cross_entropy(logits,ydev)
loss

tensor(2.1419, grad_fn=<NllLossBackward0>)

In [351]:
for _ in range(20):
    out = []
    context = [0]*block_size
    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1,-1)@W1+b1)
        logits = h@W2+b2
        propbs = F.softmax(logits)
        ix = torch.multinomial(propbs, num_samples=1, generator=g).item()
        context = context[1:]+[ix]
        out.append(ix)
        if ix ==0:
            break
    print("".join(itos[i] for i in out))

andro.
jozegh.
louriana.
gibser.
casei.
baylyn.
jantiesetriora.
nah.
nakiryer.
keoil.
daqishinsa.
colaniel.
braislee.
elyn.
ell.
chookharuhaad.
paith.
shrea.
arbeth.
izaezayla.


/var/folders/tf/0hsnm4zx7w998xhnmj8tqn3c0000gn/T/ipykernel_35585/1972365557.py:8: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  propbs = F.softmax(logits)
